## Feature Engineering

In this section we are going to create the features that the model is going to use.

We already have BTC data extracted from MT5 that we used in previous ML projects, so we are going to skip the data data extraction block.

### Loading and cleaning data

In this case the data is a .csv file inside the "data" folder.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load raw MT5 data — BTC/USD 5M
df = pd.read_csv(
    '../data/BTCUSD_M5_202301020005_202605082250.csv',
    sep='\t',
    parse_dates=[['<DATE>', '<TIME>']],
    index_col=0
)

# Clean column names
df.columns = df.columns.str.replace('<', '').str.replace('>', '')
df.index.name = 'datetime'

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst rows:")
print(df.head())
print("\nPeriod:")
print(f"  From: {df.index[0]}")
print(f"  To:   {df.index[-1]}")

C:\Users\mad52\AppData\Local\Temp\ipykernel_37288\2025509583.py:6: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv(


Shape: (346178, 7)

Columns: ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']

First rows:
                         OPEN      HIGH       LOW     CLOSE  TICKVOL  VOL  \
datetime                                                                    
2023-01-02 00:05:00  16599.62  16605.81  16597.26  16599.79      748    0   
2023-01-02 00:10:00  16599.86  16600.26  16594.76  16595.43      203    0   
2023-01-02 00:15:00  16595.47  16595.94  16584.76  16589.17      480    0   
2023-01-02 00:20:00  16589.91  16597.31  16589.01  16592.44      251    0   
2023-01-02 00:25:00  16592.43  16595.63  16589.76  16591.49      242    0   

                     SPREAD  
datetime                     
2023-01-02 00:05:00     197  
2023-01-02 00:10:00     197  
2023-01-02 00:15:00     197  
2023-01-02 00:20:00     197  
2023-01-02 00:25:00     197  

Period:
  From: 2023-01-02 00:05:00
  To:   2026-05-08 22:50:00


### Building CVD and volume features

In this section we are going to create 3 new features:

* Volume delta
Directional volume by candle, green candle at close means positive delta

* CVD Cumulative Volume Delta
Sum of volume delta since the beginning of the dataset, measures who is in control of the market historically.

* CVD_50 , CVD_100 - CVD Rolling
Measures buying/selling pressure on recent windows.

* Vol_ma_20 - Volume moving average 
Volume average of las 20 candles (1.5h), reference of "normal" volume in that period.

* Vol_ratio
Measures how much the actual volume is higher than the average volume (1= average vol, 2= double the average vol and so on)

* vol_zscore
Measures how many standard deviations is the current volume away from the recent mean.

* return_1 and return_12
return_1 last candle movement (%)
return_12 las hour candle movement (%)

Necessary to detect divergence and anomalous price movements.

* candle_size
Classifies body candle size in 3 types, normal, big, huge. In confluence with vol_ratio signals a strong institutional activity.

* price_direction and cvd_direction
Compares the actual price with the price in the past, to see if it has gone up or down.

* divergence
Compares price versus CVD (cummulative volume delta) to look for a diverge

In [4]:
# ── Volume Delta ──────────────────────────────────────────────────────────────
# Directional volume — positive if bullish candle, negative if bearish
df['vol_delta'] = np.where(
    df['CLOSE'] >= df['OPEN'],
     df['TICKVOL'],   # bullish candle → buying pressure
    -df['TICKVOL']    # bearish candle → selling pressure
)

# ── CVD — Cumulative Volume Delta ─────────────────────────────────────────────
df['CVD'] = df['vol_delta'].cumsum()

# ── Rolling CVD (50 bars = ~4 hours) ─────────────────────────────────────────
df['CVD_50']  = df['vol_delta'].rolling(50).sum()
df['CVD_100'] = df['vol_delta'].rolling(100).sum()

# ── Volume features ───────────────────────────────────────────────────────────
df['vol_ma_20']    = df['TICKVOL'].rolling(20).mean()
df['vol_ratio']    = df['TICKVOL'] / df['vol_ma_20']  # volume vs average
df['vol_zscore']   = (df['TICKVOL'] - df['vol_ma_20']) / df['TICKVOL'].rolling(20).std()

# ── Price features ────────────────────────────────────────────────────────────
df['return_1']  = df['CLOSE'].pct_change() * 100
df['return_12'] = df['CLOSE'].pct_change(12) * 100  # 1 hour
df['candle_size'] = abs(df['CLOSE'] - df['OPEN']) / df['OPEN'] * 100

# ── CVD Divergence ────────────────────────────────────────────────────────────
# Key feature: price direction vs CVD direction
df['price_direction'] = np.sign(df['return_12'])
df['cvd_direction']   = np.sign(df['CVD_50'].diff(12))
df['divergence']      = (df['price_direction'] != df['cvd_direction']).astype(int)

# Drop NaN
df_clean = df.dropna()
print(f"Shape after dropna: {df_clean.shape}")
print(f"\nFeatures created:")
print(df_clean[['TICKVOL', 'vol_delta', 'CVD_50', 
                 'vol_ratio', 'vol_zscore', 'divergence']].tail())

Shape after dropna: (346079, 20)

Features created:
                     TICKVOL  vol_delta   CVD_50  vol_ratio  vol_zscore  \
datetime                                                                  
2026-05-08 22:30:00     4537       4537  15664.0   0.893057   -0.647677   
2026-05-08 22:35:00     3934      -3934   6231.0   0.779952   -1.271292   
2026-05-08 22:40:00     4866       4866   5092.0   0.975972   -0.142241   
2026-05-08 22:45:00     3656       3656   3964.0   0.738504   -1.465250   
2026-05-08 22:50:00     3708       3708   2533.0   0.750790   -1.369028   

                     divergence  
datetime                         
2026-05-08 22:30:00           1  
2026-05-08 22:35:00           1  
2026-05-08 22:40:00           1  
2026-05-08 22:45:00           1  
2026-05-08 22:50:00           1  
